# Readable packing ablation: inference only
Create a NEW PRIVATE notebook. Attach a PRIVATE Dataset containing ONLY readable_pack.json. No adapter or previous output attachment is required. Enable Internet, GPU T4 x2 and the HF_TOKEN Kaggle Secret. Only GPU 0 is used. Never put credentials in cells or data.
Four first-attempt greedy generations: two retrospectively selected TRAINING paragraphs, each with frozen offset/JSON and readable bindings/direct-plan conditions. The cases are the hierarchy merge and market-efficiency finding/presumption split. No new annotations, development or reserved test data.
A exactly reuses the earlier edit-guided prompt. B shows labeled source sentences, verbatim source claim/protected excerpts and the same edit slots as direct sentence instructions. Facts, claims, relationships, operation assignments, model and decoding stay fixed. Binding display and plan display change together; this does not isolate their separate effects. References and target wording are evaluation-only, never generator inputs.
Frozen base Gemma 3 4B; pinned revisions, NF4/FP32 on one T4. 768 output tokens and 4096 total-token limit. All prompts are tokenizer-audited before Gemma weights load; no truncation. No training, adapters, retries, repairs or reference fallback. Empty/cutoff attempts remain failures.
Gemma unloads before the smaller NLI model loads. Save full semantic and claim-localization evidence, copying, style proxies, tokens, runtime and paired results. Check whether the actual merge/split is realized; counts alone are insufficient. Minor claim drift can be acceptable for worthwhile style gains; binary NLI flags do not establish severity. This is a tiny oracle format diagnostic, not generalization or learned mapping.
Download /kaggle/working/readable_packing_ablation_v1/ in full and keep it private. Share the notebook slug/version after completion. Phase messages and heartbeats show progress without credential values.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"


In [ ]:
%pip install -q "transformers==5.20.1" "accelerate==1.14.0" "bitsandbytes==0.50.2" "huggingface-hub==1.33.0" "sentencepiece>=0.2"


In [ ]:
import base64, json, sys, zlib
from pathlib import Path
runtime = Path("/kaggle/working/_readable_packing_runtime")
files = json.loads(zlib.decompress(base64.b64decode('...')).decode("utf-8"))
for name, source in files.items():
    path = runtime / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source, encoding="utf-8")
sys.path.insert(0, str(runtime / "src"))


In [ ]:
import importlib
experiment = importlib.import_module("fama_style.rhetoric.readable_packing")
matches = list(Path("/kaggle/input").rglob("readable_pack.json"))
if len(matches) != 1:
    raise ValueError("Attach exactly one private readable_pack.json")
arguments = ["--pack", str(matches[0]), "--expected-pack-sha", '56071e85359cc84c899949ce0352c0a36f57d111c49123f8da6c6756a7f22853']
experiment.main(arguments + ["--output-dir", "/kaggle/working/readable_packing_preflight"])


In [ ]:
experiment.main(arguments + ["--execute-inference", "--output-dir", "/kaggle/working/readable_packing_ablation_v1"])
